# Entity Typing, Deduplication & Balanced Scoring

Fix three issues with the initial concept extraction:
1. **Entity typing** — 65% of concepts classified as UNKNOWN. Use category signals + larger spaCy model to fix.
2. **Deduplication** — "(pete) rose" and "pete rose" are separate. Clean and merge.
3. **Geographic dominance** — Top 50 is almost all places. Score within entity types for a balanced list.

In [ ]:
import pandas as pd
import numpy as np
import spacy
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Use the larger model for better NER
nlp = spacy.load("en_core_web_lg")
print(f"spaCy model: {nlp.meta['name']} v{nlp.meta['version']}")

In [ ]:
# Load raw data
df = pd.read_csv(
    "../data/jeopardy_clue_dataset/combined_season1-41.tsv",
    sep="\t",
    dtype={"clue_value": "Int64", "daily_double_value": "Int64"},
    parse_dates=["air_date"],
    low_memory=False,
)
df["season"] = df["air_date"].apply(lambda d: d.year if d.month >= 9 else d.year - 1)
df = df.dropna(subset=["question"])
print(f"Loaded {len(df):,} clues")

## 1. Improved Response Normalization

Better cleaning than notebook 02:
- Handle parenthetical name hints: `(Pete) Rose` -> `Pete Rose`
- Handle `or` alternatives: `nitrous oxide (or laughing gas)` -> `nitrous oxide`
- Handle ampersand and slash variations
- Strip quotes

In [ ]:
def normalize_response(text):
    """Normalize a Jeopardy response to a canonical concept form."""
    if not isinstance(text, str):
        return ""
    s = text.strip()
    
    # Unwrap parenthetical name hints: (Pete) Rose -> Pete Rose
    s = re.sub(r"\(([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\)", r"\1", s)
    
    # Remove trailing parenthetical alternatives: "X (or Y)" -> "X"
    s = re.sub(r"\s*\(or\s+[^)]+\)\s*$", "", s)
    # Remove trailing parenthetical clarifiers: "Mercury (the planet)" -> "Mercury"
    # But keep parenthetical parts that are part of the name
    
    # Strip quotes
    s = s.strip('"\'')
    
    # Lowercase
    s = s.lower().strip()
    
    # Remove Jeopardy phrasing if present
    s = re.sub(r"^(what|who|where|when)\s+(is|are|was|were|does)\s+", "", s)
    
    # Strip leading articles
    s = re.sub(r"^(the|a|an)\s+", "", s)
    
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    
    return s

df["concept"] = df["question"].apply(normalize_response)

# Also keep a title-case version for NER (spaCy works better with proper casing)
def to_title_for_ner(text):
    """Convert to title case for NER, preserving some patterns."""
    if not isinstance(text, str):
        return ""
    s = text.strip()
    # Unwrap parenthetical name hints
    s = re.sub(r"\(([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\)", r"\1", s)
    s = re.sub(r"\s*\(or\s+[^)]+\)\s*$", "", s)
    s = s.strip('"\'')
    s = re.sub(r"^(the|a|an|The|A|An)\s+", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["concept_title"] = df["question"].apply(to_title_for_ner)

n_unique = df["concept"].nunique()
print(f"Unique normalized concepts: {n_unique:,}")

# Show some examples where normalization changed things
changed = df[df["question"].str.lower().str.strip() != df["concept"]][["question", "concept"]].drop_duplicates("question").head(15)
print("\nNormalization examples (where something changed):")
for _, row in changed.iterrows():
    print(f"  {row['question']!r:45s} -> {row['concept']!r}")

## 2. Category-Based Entity Typing

The 56k unique Jeopardy categories contain strong signals about what entity type the answer is. Map category keywords to entity types.

In [ ]:
# Category keyword -> entity type mapping
# Order matters: more specific patterns first
CATEGORY_TYPE_RULES = [
    # PERSON indicators
    ("PERSON", [
        r"\bpresident", r"\bking(?:s)?\b", r"\bqueen(?:s)?\b", r"\bauthor",
        r"\bwriter", r"\bpoet(?:s)?\b", r"\bactor", r"\bactress",
        r"\bcomposer", r"\bscientist", r"\binventor", r"\bexplorer",
        r"\bartist", r"\bpainter", r"\bmusician", r"\bsinger",
        r"\bdirector", r"\bphilosopher", r"\bgeneral(?:s)?\b",
        r"\bleader", r"\bpharaoh", r"\bemperor", r"\bdictator",
        r"\bpeople\b", r"\bwho\b", r"\bcelebrit", r"\bfamous\b",
        r"\bhero(?:es)?\b", r"\bvillain", r"\bsaint(?:s)?\b",
        r"\bnovel(?:ist)", r"\bplaywright", r"\bastrona",
        r"\bfirst lad", r"\bfounding father", r"\bprime minister",
        r"\bpope(?:s)?\b", r"\bprophet", r"\bapostle",
        r"\bname'?s the same\b", r"\bchannel\b.*\bname",
    ]),
    # WORK_OF_ART indicators
    ("WORK_OF_ART", [
        r"\bnovel(?:s)?\b", r"\bbook(?:s)?\b", r"\bfilm(?:s)?\b",
        r"\bmovie(?:s)?\b", r"\bplay(?:s)?\b", r"\bsong(?:s)?\b",
        r"\bpainting(?:s)?\b", r"\bpoem(?:s)?\b", r"\bopera(?:s)?\b",
        r"\bmusical(?:s)?\b", r"\bsymphon", r"\balbum",
        r"\btv show", r"\bsitcom", r"\bbestseller", r"\bbest seller",
        r"\baward.?winning", r"\b(?:oscar|emmy|tony|grammy).?winner",
        r"\btitle\b",
    ]),
    # GPE indicators (countries, states, cities)
    ("GPE", [
        r"\bcountr(?:y|ies)\b", r"\bnation(?:s)?\b", r"\bstate(?:s)?\b",
        r"\bcit(?:y|ies)\b", r"\bcapital(?:s)?\b",
        r"\bu\.?s\.? (?:cit|state|geog)", r"\bamerican (?:cit|state|geog)",
        r"\bworld (?:cit|capital|geog)", r"\beuropean (?:cit|capital|geog)",
        r"\bgeograph",
    ]),
    # LOC indicators (natural features)
    ("LOC", [
        r"\briver(?:s)?\b", r"\blake(?:s)?\b", r"\bmountain(?:s)?\b",
        r"\bisland(?:s)?\b", r"\bocean(?:s)?\b", r"\bsea(?:s)?\b",
        r"\bbod(?:y|ies) of water\b", r"\bcontinent",
        r"\bdesert(?:s)?\b", r"\bvolcano",
    ]),
    # ORG indicators
    ("ORG", [
        r"\bbusiness", r"\bcompan(?:y|ies)\b", r"\bcorporat",
        r"\bcollege(?:s)?\b", r"\buniversit", r"\borganiz",
        r"\bteam(?:s)?\b", r"\bbrand(?:s)?\b", r"\bindustr",
    ]),
    # EVENT indicators
    ("EVENT", [
        r"\bwar(?:s)?\b", r"\bbattle(?:s)?\b",
        r"\bcivil war\b", r"\bworld war\b", r"\brevolution",
    ]),
    # NORP indicators (nationality, religion, political group)
    ("NORP", [
        r"\breligion(?:s)?\b", r"\bmytholog",
    ]),
    # CONCEPT indicators (science, vocabulary, etc.)
    ("CONCEPT", [
        r"\bscience\b", r"\bbiology\b", r"\bchemistry\b", r"\bphysics\b",
        r"\bastronom", r"\bmedicine\b", r"\banatom", r"\bmath",
        r"\bvocab", r"\bword(?:s)?\b", r"\blanguage(?:s)?\b",
        r"\bfood(?:s)?\b", r"\bcuisine", r"\bcooking\b",
        r"\banimal(?:s)?\b", r"\bmammal", r"\bbird(?:s)?\b", r"\bfish\b",
        r"\binvent", r"\btransport", r"\btechnolog",
        r"\blaw\b", r"\blegal\b",  r"\beconom",
        r"\bspelling\b", r"\bgrammar\b", r"\bletter(?:s)?\b",
        r"\brhyme", r"\bsport(?:s)?\b", r"\bgame(?:s)?\b",
        r"\bfashion\b", r"\bcloth", r"\bcolor(?:s)?\b",
        r"\bflower(?:s)?\b", r"\btree(?:s)?\b", r"\bplant(?:s)?\b",
        r"\bweather\b", r"\bgeolog", r"\belement",
        r"\bbefore & after\b", r"\bhodgepodge\b", r"\bpotpourri\b",
        r"\bgrab bag\b", r"\bmiscell",
    ]),
]

def classify_by_category(category):
    """Return the most likely entity type based on category name."""
    cat_lower = category.lower()
    for entity_type, patterns in CATEGORY_TYPE_RULES:
        for pattern in patterns:
            if re.search(pattern, cat_lower):
                return entity_type
    return None  # No match

# Apply to all rows
df["cat_entity_type"] = df["category"].apply(classify_by_category)

cat_classified = df["cat_entity_type"].notna().sum()
print(f"Category-classified: {cat_classified:,} / {len(df):,} ({100*cat_classified/len(df):.1f}%)")
print(f"\nCategory type distribution:")
print(df["cat_entity_type"].value_counts().to_string())

## 3. Re-run NER with Larger Model

The `en_core_web_lg` model (300MB) has much better entity recognition than `en_core_web_sm` (12MB), especially for lowercase and less common entities. Run it on unique concepts using title-case versions for better recognition.

In [ ]:
# Build mapping: normalized concept -> best title-case version (most frequent original form)
concept_to_title = (
    df.groupby("concept")["concept_title"]
    .agg(lambda x: x.value_counts().index[0])  # most frequent original form
    .to_dict()
)

unique_concepts = df["concept"].value_counts().index.tolist()
title_texts = [concept_to_title.get(c, c) for c in unique_concepts]

print(f"Running NER (en_core_web_lg) on {len(unique_concepts):,} unique concepts...")

ner_entity_types = {}
batch_size = 1000

for i in range(0, len(title_texts), batch_size):
    batch_titles = title_texts[i:i+batch_size]
    batch_concepts = unique_concepts[i:i+batch_size]
    for doc, concept in zip(nlp.pipe(batch_titles, disable=["parser", "lemmatizer"]), batch_concepts):
        if doc.ents:
            best_ent = max(doc.ents, key=lambda e: len(e.text))
            ner_entity_types[concept] = best_ent.label_
        else:
            ner_entity_types[concept] = "UNKNOWN"
    if (i // batch_size) % 25 == 0:
        print(f"  Processed {min(i+batch_size, len(unique_concepts)):,} / {len(unique_concepts):,}")

print(f"\nNER type distribution (en_core_web_lg):")
ner_dist = Counter(ner_entity_types.values())
for t, c in ner_dist.most_common():
    print(f"  {t:20s} {c:>8,}")

## 4. Merge Entity Type Sources

Combine category-based typing and NER typing. Priority:
1. If a concept's **majority category type** agrees with NER -> high confidence
2. If NER says UNKNOWN but category says something -> use category
3. If category is ambiguous but NER has a type -> use NER
4. If both are UNKNOWN -> stays UNKNOWN

In [ ]:
# For each concept, get the majority category-based type
# (a concept may appear under multiple categories)
concept_cat_types = (
    df[df["cat_entity_type"].notna()]
    .groupby("concept")["cat_entity_type"]
    .agg(lambda x: x.value_counts().index[0])  # most common category type
    .to_dict()
)

# Also compute category type agreement ratio
concept_cat_agreement = (
    df[df["cat_entity_type"].notna()]
    .groupby("concept")["cat_entity_type"]
    .agg(lambda x: x.value_counts().iloc[0] / len(x))  # fraction agreeing on top type
    .to_dict()
)

# NER types to treat as "resolved" (not noise)
RESOLVED_NER_TYPES = {"PERSON", "GPE", "LOC", "ORG", "NORP", "FAC", "WORK_OF_ART", "EVENT", "PRODUCT", "LAW", "LANGUAGE"}

def merge_entity_type(concept):
    """Merge NER and category-based entity types."""
    ner_type = ner_entity_types.get(concept, "UNKNOWN")
    cat_type = concept_cat_types.get(concept)
    cat_agree = concept_cat_agreement.get(concept, 0)
    
    ner_resolved = ner_type in RESOLVED_NER_TYPES
    cat_resolved = cat_type is not None and cat_type != "CONCEPT"
    
    # Both agree -> high confidence
    if ner_resolved and cat_resolved:
        # Map NER types to our taxonomy for comparison
        # If they roughly agree, use the more specific one
        return cat_type if cat_agree > 0.6 else ner_type
    
    # Only NER resolved
    if ner_resolved:
        return ner_type
    
    # Only category resolved
    if cat_type is not None:
        return cat_type
    
    # Neither resolved
    return "UNKNOWN"

# Apply to all unique concepts
merged_types = {c: merge_entity_type(c) for c in unique_concepts}

print("Merged entity type distribution:")
merged_dist = Counter(merged_types.values())
for t, c in merged_dist.most_common():
    print(f"  {t:20s} {c:>8,}")

unknown_pct = merged_dist.get("UNKNOWN", 0) / len(merged_types) * 100
print(f"\nUNKNOWN reduced from 65% to {unknown_pct:.1f}%")

## 5. Deduplication

Merge variant forms of the same concept:
- Case/article variants already handled by normalization
- Remaining issues: abbreviations, name variants, synonyms

In [ ]:
# Rebuild concept stats with merged types
concept_stats = df.groupby("concept").agg(
    frequency=("concept", "size"),
    seasons_appeared=("season", "nunique"),
    category_breadth=("category", "nunique"),
    avg_clue_value=("clue_value", "mean"),
    first_seen=("air_date", "min"),
    last_seen=("air_date", "max"),
).reset_index()

concept_stats["entity_type"] = concept_stats["concept"].map(merged_types)
concept_stats["display_name"] = concept_stats["concept"].map(concept_to_title)

# Sort by frequency to establish canonical forms
concept_stats = concept_stats.sort_values("frequency", ascending=False).reset_index(drop=True)

print(f"Before dedup: {len(concept_stats):,} concepts")

# --- Dedup rules ---

# 1. Build a simple alias map for known duplicates
# These are concepts that should merge (canonical <- variant)
# We'll detect them by finding concepts that are substrings of each other
# within the same entity type

# Group concepts by entity type for dedup within type
type_groups = concept_stats.groupby("entity_type")["concept"].apply(list).to_dict()

# For efficiency, only dedup concepts with frequency >= 3
freq_concepts = concept_stats[concept_stats["frequency"] >= 3].copy()
print(f"Concepts with freq >= 3: {len(freq_concepts):,}")

# Find potential duplicates: same entity type, one is a suffix of the other
# e.g., "franklin roosevelt" and "franklin d. roosevelt"
# or "mark twain" and "twain"
alias_map = {}  # variant -> canonical

for etype, concepts in type_groups.items():
    if etype == "UNKNOWN" or etype == "CONCEPT":
        continue
    # Only look at concepts with freq >= 3
    concepts = [c for c in concepts if c in freq_concepts["concept"].values]
    # Sort by frequency (descending) so canonical form is the most frequent
    freq_lookup = dict(zip(concept_stats["concept"], concept_stats["frequency"]))
    concepts.sort(key=lambda c: -freq_lookup.get(c, 0))
    
    # For each concept, check if a shorter variant is a suffix match
    seen = set()
    for c in concepts:
        if c in alias_map:
            continue
        words = c.split()
        if len(words) >= 2:
            # Check if last name alone exists as a separate concept
            last = words[-1]
            if last in freq_lookup and last != c and last not in alias_map:
                # Only merge if the short form has the same entity type
                if merged_types.get(last) == etype:
                    # Don't merge common words that happen to match
                    if freq_lookup.get(last, 0) < freq_lookup.get(c, 0):
                        alias_map[last] = c

print(f"\nFound {len(alias_map):,} alias mappings")
if alias_map:
    # Show first 20
    items = list(alias_map.items())[:20]
    print("Sample aliases:")
    for variant, canonical in items:
        print(f"  {variant!r:25s} -> {canonical!r}")

In [ ]:
# Apply alias map to merge duplicates
df["concept_deduped"] = df["concept"].map(lambda c: alias_map.get(c, c))

# Rebuild concept stats with deduped concepts
concept_stats_deduped = df.groupby("concept_deduped").agg(
    frequency=("concept_deduped", "size"),
    seasons_appeared=("season", "nunique"),
    category_breadth=("category", "nunique"),
    avg_clue_value=("clue_value", "mean"),
    first_seen=("air_date", "min"),
    last_seen=("air_date", "max"),
).reset_index()

concept_stats_deduped.rename(columns={"concept_deduped": "concept"}, inplace=True)
concept_stats_deduped["entity_type"] = concept_stats_deduped["concept"].map(merged_types)
concept_stats_deduped["display_name"] = concept_stats_deduped["concept"].map(concept_to_title)

concept_stats_deduped = concept_stats_deduped.sort_values("frequency", ascending=False).reset_index(drop=True)

print(f"After dedup: {len(concept_stats_deduped):,} concepts")
print(f"Reduced by: {len(concept_stats) - len(concept_stats_deduped):,}")

## 6. Entity-Type-Aware Importance Scoring

Score within entity types, then combine into a balanced overall list. This ensures PERSONs, WORK_OF_ARTs, etc. aren't drowned out by places.

In [ ]:
def min_max_normalize(series):
    rng = series.max() - series.min()
    if rng == 0:
        return series * 0
    return (series - series.min()) / rng

cs = concept_stats_deduped.copy()

# Global importance score (same formula as before)
cs["freq_score"] = min_max_normalize(np.log1p(cs["frequency"]))
cs["season_score"] = min_max_normalize(cs["seasons_appeared"])
cs["breadth_score"] = min_max_normalize(np.log1p(cs["category_breadth"]))

cs["importance_global"] = (
    0.40 * cs["freq_score"] +
    0.35 * cs["season_score"] +
    0.25 * cs["breadth_score"]
)

# Within-type importance score (normalized within each entity type)
cs["importance_within_type"] = 0.0
for etype in cs["entity_type"].unique():
    mask = cs["entity_type"] == etype
    subset = cs.loc[mask].copy()
    if len(subset) < 2:
        continue
    subset["freq_s"] = min_max_normalize(np.log1p(subset["frequency"]))
    subset["season_s"] = min_max_normalize(subset["seasons_appeared"])
    subset["breadth_s"] = min_max_normalize(np.log1p(subset["category_breadth"]))
    subset["imp"] = 0.40 * subset["freq_s"] + 0.35 * subset["season_s"] + 0.25 * subset["breadth_s"]
    cs.loc[mask, "importance_within_type"] = subset["imp"].values

cs = cs.sort_values("importance_global", ascending=False).reset_index(drop=True)
cs.index += 1
cs.index.name = "global_rank"

print("Top 50 by global importance:")
print(cs.head(50)[["concept", "display_name", "entity_type", "frequency", "seasons_appeared", "category_breadth", "importance_global"]].to_string())

In [ ]:
# Top concepts PER entity type
print("=" * 80)
print("TOP 20 CONCEPTS BY ENTITY TYPE")
print("=" * 80)

for etype in ["PERSON", "GPE", "LOC", "ORG", "WORK_OF_ART", "EVENT", "NORP", "CONCEPT", "UNKNOWN"]:
    subset = cs[cs["entity_type"] == etype].head(20)
    if len(subset) == 0:
        continue
    print(f"\n--- {etype} ({(cs['entity_type'] == etype).sum():,} total) ---")
    print(subset[["concept", "display_name", "frequency", "seasons_appeared", "importance_within_type"]].to_string())

In [ ]:
# Visualize top 15 per major type
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

type_colors = {
    "GPE": "#2196F3", "PERSON": "#FF9800", "ORG": "#4CAF50",
    "WORK_OF_ART": "#F44336", "CONCEPT": "#607D8B", "EVENT": "#795548",
}

for ax, etype in zip(axes.flat, ["PERSON", "GPE", "CONCEPT", "WORK_OF_ART", "ORG", "EVENT"]):
    subset = cs[cs["entity_type"] == etype].head(15)
    if len(subset) == 0:
        ax.set_visible(False)
        continue
    color = type_colors.get(etype, "#9E9E9E")
    ax.barh(range(len(subset)), subset["importance_within_type"], color=color, edgecolor="none")
    ax.set_yticks(range(len(subset)))
    display = subset["display_name"].fillna(subset["concept"]).tolist()
    ax.set_yticklabels(display, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f"Top 15: {etype}", fontweight="bold")
    ax.set_xlabel("Importance (within type)")

plt.suptitle("Cultural Literacy Concepts by Entity Type", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Balanced Composite List

Build a "Top N" list that draws proportionally from each entity type, reflecting the breadth of cultural literacy.

In [ ]:
# Build a balanced top-200 list
# Allocation: roughly proportional to type frequency but with floors
TYPE_ALLOCATION = {
    "PERSON": 50,
    "GPE": 40,
    "CONCEPT": 35,
    "WORK_OF_ART": 25,
    "ORG": 15,
    "EVENT": 10,
    "LOC": 10,
    "NORP": 10,
    "UNKNOWN": 5,
}

balanced_list = []
for etype, n in TYPE_ALLOCATION.items():
    subset = cs[cs["entity_type"] == etype].head(n)
    balanced_list.append(subset)

balanced = pd.concat(balanced_list).sort_values("importance_global", ascending=False).reset_index(drop=True)
balanced.index += 1
balanced.index.name = "rank"

print(f"Balanced composite list: {len(balanced)} concepts")
print(f"\nType distribution:")
print(balanced["entity_type"].value_counts().to_string())
print(f"\nFull list:")
print(balanced[["concept", "display_name", "entity_type", "frequency", "seasons_appeared", "importance_global"]].to_string())

## 8. Export Improved Data

In [ ]:
# Save full improved concept table
output_path = "../data/jeopardy_concepts_v2.csv"
export_cols = [
    "concept", "display_name", "entity_type", "frequency", "seasons_appeared",
    "category_breadth", "avg_clue_value", "first_seen", "last_seen",
    "importance_global", "importance_within_type",
]
cs[export_cols].to_csv(output_path, index=True)
print(f"Saved {len(cs):,} concepts to {output_path}")

# Save balanced top-200 list
balanced_path = "../data/jeopardy_top200_balanced.csv"
balanced[export_cols].to_csv(balanced_path, index=True)
print(f"Saved balanced top-200 to {balanced_path}")

# Summary
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total concepts: {len(cs):,}")
print(f"Entity type coverage: {100*(1 - (cs['entity_type']=='UNKNOWN').mean()):.1f}%")
print(f"\nConcepts by type (freq >= 5):")
freq5 = cs[cs["frequency"] >= 5]
print(freq5["entity_type"].value_counts().to_string())
print(f"\nConcepts appearing in 20+ seasons: {(cs['seasons_appeared'] >= 20).sum():,}")